# 击中击不中变换实验：先击中，再击不中（学生练习版）

本 Notebook 是学生练习版：随机图案生成、Hit/Miss 模板定义和可视化已经保留；Hit 匹配、Miss 判断和完整变换函数被改为 `TODO`。

本实验重新生成击中击不中变换程序，采用更清晰的三步流程：

1. **随机生成一组黑白二值图案**。
2. **确定一组前景击中模板 Hit template 和背景击不中模板 Miss template**。
3. **先用 Hit template 查看有多少位置满足前景击中条件**。
4. **再在这些候选位置中检查 Miss template，看有多少同时满足背景击不中条件**。

这样可以清楚观察：击中击不中变换并不是只做一次模板匹配，而是先找“前景形状符合”的候选点，再用“背景约束”进一步筛选。

## 1. 相关背景知识

### 1.1 二值图像

本实验使用黑白二值图像：

- `1`：白色前景。
- `0`：黑色背景。

### 1.2 前景击中模板 Hit template

Hit template 规定哪些位置必须是前景。

例如：

```text
[0, 1, 0]
[0, 1, 1]
[0, 0, 0]
```

表示：中心点、上方点、右侧点必须是前景。

### 1.3 背景击不中模板 Miss template

Miss template 规定哪些位置必须是背景。

例如：

```text
[0, 0, 0]
[1, 0, 0]
[0, 1, 0]
```

表示：中心左侧和下方必须是背景。

### 1.4 分阶段理解击中击不中变换

击中击不中变换可以分成两个判断阶段：

1. **Hit 阶段**：只检查前景模板，找出满足前景形状的候选位置。
2. **Miss 阶段**：在 Hit 阶段候选位置中继续检查背景模板。

最终结果为：

`Hit-or-Miss = Hit 匹配结果 ∩ Miss 匹配结果`

也就是说，最终匹配点必须同时满足：

- Hit 模板要求的位置都是前景。
- Miss 模板要求的位置都是背景。

## 2. 随机生成实验图案

为了让实验每次都有一定变化，本实验会随机生成一组由小图案组成的二值图像。

图像中随机放置若干 `3 × 3` 局部图案，其中包括：

- 满足 Hit 和 Miss 的目标图案。
- 只满足 Hit 但不满足 Miss 的干扰图案。
- 完全不满足 Hit 的随机图案。

这样可以观察背景击不中模板如何从 Hit 候选点中进一步筛选真正目标。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def show_binary_image(image, title="", ax=None, show_grid=False):
    """显示二值图像。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
    if show_grid:
        ax.set_xticks(np.arange(-0.5, image.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, image.shape[0], 1), minor=True)
        ax.grid(which="minor", color="#dddddd", linewidth=0.35)


def overlay_points(image, points, color=(1.0, 0.0, 0.0)):
    """把匹配点叠加到原图上。"""
    rgb = np.dstack([image, image, image]).astype(float)
    for y, x in points:
        rgb[y, x] = color
    return rgb


def coordinates_from_mask(mask):
    """把二值匹配图转换为坐标列表。"""
    ys, xs = np.where(mask == 1)
    return list(zip(ys.tolist(), xs.tolist()))

## 3. 定义 Hit 模板和 Miss 模板

本实验检测一种“向上、向右延伸的 L 形角点”。

### Hit 模板要求

下面三个位置必须是前景：

- 中心点
- 上方点
- 右侧点

### Miss 模板要求

下面两个位置必须是背景：

- 左侧点
- 下方点

也就是说，这个模板要寻找一种“上方和右侧有连接，但左侧和下方没有连接”的局部结构。

In [ ]:
hit_template = np.array([
    [0, 1, 0],
    [0, 1, 1],
    [0, 0, 0],
], dtype=np.uint8)

miss_template = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
], dtype=np.uint8)


def show_hit_and_miss_templates(hit, miss):
    """显示前景击中模板和背景击不中模板。"""
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))

    axes[0].imshow(hit, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("前景击中模板 Hit\n白色=必须为前景")

    axes[1].imshow(1 - miss, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("背景击不中模板 Miss\n黑色=必须为背景")

    combined = np.full(hit.shape, 0.5)
    combined[hit == 1] = 1.0
    combined[miss == 1] = 0.0
    axes[2].imshow(combined, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("合成观察\n白=前景 黑=背景 灰=不关心")

    for ax in axes:
        ax.set_xticks(np.arange(-0.5, hit.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, hit.shape[0], 1), minor=True)
        ax.grid(which="minor", color="#cc3333", linewidth=1.0)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    plt.tight_layout()
    plt.show()


show_hit_and_miss_templates(hit_template, miss_template)

## 4. 随机生成包含目标和干扰的二值图像

下面的函数会把若干 `3 × 3` 小图案随机放置到大图像中。

其中：

- `target_patch`：同时满足 Hit 和 Miss。
- `hit_only_patch`：满足 Hit，但不满足 Miss。
- `random_patch`：随机干扰图案。

程序会记录每个图案的中心位置，便于观察最终检测结果是否合理。

In [ ]:
target_patch = np.array([
    [0, 1, 0],
    [0, 1, 1],
    [0, 0, 0],
], dtype=np.uint8)

hit_only_patch = np.array([
    [0, 1, 0],
    [1, 1, 1],
    [0, 1, 0],
], dtype=np.uint8)


def can_place_patch(occupied, top, left, patch_size=3, margin=1):
    """判断某个 patch 是否可以放置，避免不同小图案重叠太近。"""
    y1 = max(0, top - margin)
    y2 = min(occupied.shape[0], top + patch_size + margin)
    x1 = max(0, left - margin)
    x2 = min(occupied.shape[1], left + patch_size + margin)
    return not occupied[y1:y2, x1:x2].any()


def place_patch(image, occupied, patch, center, label, records):
    """把小图案放入图像，并记录中心点和类型。"""
    cy, cx = center
    top, left = cy - 1, cx - 1
    image[top:top + 3, left:left + 3] = patch
    occupied[top:top + 3, left:left + 3] = 1
    records.append({"center": center, "label": label})


def generate_random_pattern_image(height=72, width=96, target_count=8, hit_only_count=8, random_count=12):
    """随机生成击中击不中实验图像。"""
    image = np.zeros((height, width), dtype=np.uint8)
    occupied = np.zeros_like(image, dtype=np.uint8)
    records = []

    patches = (
        [(target_patch, "target")] * target_count
        + [(hit_only_patch, "hit_only")] * hit_only_count
    )

    for _ in range(random_count):
        patch = (np.random.random((3, 3)) > 0.65).astype(np.uint8)
        if patch.sum() == 0:
            patch[1, 1] = 1
        patches.append((patch, "random"))

    np.random.shuffle(patches)

    for patch, label in patches:
        placed = False
        for _ in range(500):
            cy = np.random.randint(2, height - 2)
            cx = np.random.randint(2, width - 2)
            top, left = cy - 1, cx - 1
            if can_place_patch(occupied, top, left):
                place_patch(image, occupied, patch, (cy, cx), label, records)
                placed = True
                break
        if not placed:
            print(f"警告：{label} 图案未能成功放置")

    return image, records


binary_image, pattern_records = generate_random_pattern_image()

print("随机图案数量统计：")
for label in ["target", "hit_only", "random"]:
    print(f"{label}: {sum(record['label'] == label for record in pattern_records)} 个")

show_binary_image(binary_image, "随机生成的黑白二值图像", show_grid=False)
plt.show()

### 5.1 算法补全步骤

请按下面顺序完成击中击不中变换：

1. 补全 `match_hit_only(image, hit)`，只检查前景击中模板，得到 Hit 候选点。
2. 运行 Hit 阶段单元，观察满足前景模板的位置数量。
3. 补全 `check_miss_at_point(image, miss, y, x)`，判断某个 Hit 候选点是否满足背景击不中模板。
4. 运行 Miss 阶段单元，观察有多少 Hit 候选点被背景条件排除。
5. 补全 `hit_or_miss_transform_stepwise(image, hit, miss)`，整合 Hit 阶段、Miss 阶段和最终匹配结果。
6. 最后与真实随机放置的 target 图案中心进行对照。

## 5. 第一步：只使用 Hit 模板查找前景击中点

第一步只检查 Hit 模板。

也就是说，只要某个位置满足：

- 中心是前景。
- 上方是前景。
- 右侧是前景。

就被认为是 Hit 候选点。

这一阶段暂时不检查左侧和下方是不是背景，因此可能会包含一些干扰点。

In [ ]:
def match_hit_only(image, hit):
    """只根据 Hit 模板匹配前景。"""
    # TODO 1：根据 hit 模板大小对 image 做 0 填充。
    # TODO 2：遍历每个像素，取出以该像素为中心的邻域 region。
    # TODO 3：找到 hit == 1 的位置，检查这些位置在 region 中是否全部为前景 1。
    # TODO 4：满足条件的位置在 result 中设为 1。
    # TODO 5：返回 Hit 阶段匹配图 result。
    raise NotImplementedError("请补全 match_hit_only 函数")


hit_matches = match_hit_only(binary_image, hit_template)
hit_points = coordinates_from_mask(hit_matches)

print(f"只满足 Hit 前景击中模板的位置数量：{len(hit_points)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_binary_image(hit_matches, "Hit 阶段匹配结果", axes[0])
axes[1].imshow(overlay_points(binary_image, hit_points, color=(1.0, 0.0, 0.0)))
axes[1].set_title("红色=满足 Hit 的候选点")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. 第二步：在 Hit 候选点中检查 Miss 背景条件

第二步只在 Hit 候选点中继续检查 Miss 模板。

Miss 模板要求：

- 左侧必须是背景。
- 下方必须是背景。

如果某个 Hit 候选点的左侧或下方不是背景，就会被排除。

In [ ]:
def check_miss_at_point(image, miss, y, x):
    """检查某个位置是否满足 Miss 背景击不中模板。"""
    # TODO 1：根据 miss 模板大小对 image 做 0 填充。
    # TODO 2：取出以 (y, x) 为中心的邻域 region。
    # TODO 3：找到 miss == 1 的位置，检查这些位置在 region 中是否全部为背景 0。
    # TODO 4：返回 True 或 False。
    raise NotImplementedError("请补全 check_miss_at_point 函数")


final_matches = np.zeros_like(binary_image, dtype=np.uint8)
rejected_by_miss = np.zeros_like(binary_image, dtype=np.uint8)

for y, x in hit_points:
    if check_miss_at_point(binary_image, miss_template, y, x):
        final_matches[y, x] = 1
    else:
        rejected_by_miss[y, x] = 1

final_points = coordinates_from_mask(final_matches)
rejected_points = coordinates_from_mask(rejected_by_miss)

print(f"Hit 候选点数量：{len(hit_points)}")
print(f"其中满足 Miss 背景击不中模板的数量：{len(final_points)}")
print(f"被 Miss 背景条件排除的数量：{len(rejected_points)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(overlay_points(binary_image, hit_points, color=(1.0, 0.0, 0.0)))
axes[0].set_title("Hit 候选点：红色")
axes[0].axis("off")

axes[1].imshow(overlay_points(binary_image, rejected_points, color=(1.0, 0.6, 0.0)))
axes[1].set_title("被 Miss 排除：橙色")
axes[1].axis("off")

axes[2].imshow(overlay_points(binary_image, final_points, color=(0.0, 1.0, 0.0)))
axes[2].set_title("最终匹配点：绿色")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 7. 完整击中击不中变换函数

根据前面的分步过程，可以把击中击不中变换封装成一个函数。

该函数会返回三个结果：

1. Hit 阶段候选匹配图。
2. 被 Miss 条件排除的候选点。
3. 最终同时满足 Hit 和 Miss 的匹配图。

In [ ]:
def hit_or_miss_transform_stepwise(image, hit, miss):
    """分步执行击中击不中变换。"""
    # TODO 1：调用 match_hit_only 得到 hit_result。
    # TODO 2：把 hit_result 转换成 Hit 候选点坐标列表。
    # TODO 3：创建 rejected_result 和 final_result。
    # TODO 4：遍历每个 Hit 候选点，调用 check_miss_at_point 检查背景条件。
    # TODO 5：满足 Miss 的点放入 final_result，否则放入 rejected_result。
    # TODO 6：返回 hit_result、rejected_result、final_result。
    raise NotImplementedError("请补全 hit_or_miss_transform_stepwise 函数")


hit_result, rejected_result, final_result = hit_or_miss_transform_stepwise(
    binary_image,
    hit_template,
    miss_template,
)

print("封装函数运行结果：")
print(f"Hit 候选点数量：{int(hit_result.sum())}")
print(f"被 Miss 排除数量：{int(rejected_result.sum())}")
print(f"最终匹配数量：{int(final_result.sum())}")

## 8. 与真实放置图案进行对照

由于本实验在生成图像时记录了每个小图案的类型，因此可以把最终检测点和真实 `target` 图案中心进行对照。

理论上：

- `target` 图案应该同时满足 Hit 和 Miss。
- `hit_only` 图案应该满足 Hit，但会被 Miss 排除。
- `random` 图案可能偶然满足，也可能不满足。

In [ ]:
target_centers = [record["center"] for record in pattern_records if record["label"] == "target"]
hit_only_centers = [record["center"] for record in pattern_records if record["label"] == "hit_only"]

target_set = set(target_centers)
final_set = set(final_points)

true_positive = len(target_set & final_set)
missed_target = len(target_set - final_set)
extra_match = len(final_set - target_set)

print(f"真实 target 图案数量：{len(target_centers)}")
print(f"最终匹配到的 target 数量：{true_positive}")
print(f"漏检 target 数量：{missed_target}")
print(f"额外匹配数量：{extra_match}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(overlay_points(binary_image, target_centers, color=(0.0, 0.4, 1.0)))
axes[0].set_title("蓝色=真实 target 中心")
axes[0].axis("off")

axes[1].imshow(overlay_points(binary_image, hit_only_centers, color=(1.0, 0.6, 0.0)))
axes[1].set_title("橙色=hit_only 干扰中心")
axes[1].axis("off")

axes[2].imshow(overlay_points(binary_image, final_points, color=(0.0, 1.0, 0.0)))
axes[2].set_title("绿色=最终 Hit-or-Miss 匹配")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 9. 实验思考

完成实验后，可以思考下面的问题：

1. 为什么只使用 Hit 模板会得到更多候选点？
2. Miss 模板在本实验中排除了哪些干扰图案？
3. 如果把 Miss 模板去掉，最终结果会发生什么变化？
4. 如果增加 Miss 模板中的背景约束，匹配点数量通常会增加还是减少？
5. 如果把 Hit 模板改得更宽松，Hit 候选点数量会怎样变化？
6. 击中击不中变换为什么适合检测特定局部结构？